In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
from tqdm import tqdm
import pandas as pd
import os
import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Availability: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

# Set environment variables
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

PyTorch version: 2.7.1+cu128
CUDA Availability: True
GPU device: NVIDIA GeForce RTX 5070 Ti
CUDA version: 12.8
Device: cuda:0


In [9]:
# ## 2. Define model parameters and functions

hist_step = 60  # Historical time steps
pred_step = 1   # Prediction time steps
input_channel = 3  # Number of input features
output_channel = 10 # Number of output features
input_data_num = hist_step * input_channel  

# Define evaluation metric functions
def median_absolute_error(y_true, y_pred):
    """Calculate the median absolute error"""
    return np.median(np.abs(y_true - y_pred))

def iqr_based_mae(y_true, y_pred):
    """Calculate IQR-based MAE"""
    errors = np.abs(y_true - y_pred).flatten()
    q1 = np.percentile(errors, 25)
    q3 = np.percentile(errors, 75)
    mask = (errors >= q1) & (errors <= q3)
    filtered_errors = errors[mask]
    if len(filtered_errors) == 0:
        return np.nan
    return np.mean(filtered_errors)

def safe_mape(y_true, y_pred, epsilon=1e-8, threshold=0.01):
    """Enhanced MAPE calculation"""
    mask = np.abs(y_true) > threshold
    if mask.sum() == 0:
        return np.nan
    ape = np.abs((y_true[mask] - y_pred[mask]) / (y_true[mask] + epsilon))
    return np.mean(ape) * 100

def calculate_max_error(y_true, y_pred):
    """Calculate the maximum error"""
    return np.max(np.abs(y_true - y_pred))

def calculate_rmse(y_true, y_pred):
    """Calculate RMSE"""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def calculate_ca(rmse, mae, r2):
    """Calculate the combined accuracy CA"""
    return 0.33 * (rmse + mae + (1 - r2))

def calculate_all_metrics(y_true, y_pred, prefix=""):
    """Calculate all evaluation metrics"""
    metrics = {}
    
    # Basic metrics
    metrics[f'{prefix}MSE'] = mean_squared_error(y_true, y_pred)
    metrics[f'{prefix}RMSE'] = np.sqrt(metrics[f'{prefix}MSE'])
    metrics[f'{prefix}MAE'] = mean_absolute_error(y_true, y_pred)
    metrics[f'{prefix}MdAE'] = median_absolute_error(y_true, y_pred)
    metrics[f'{prefix}IQR_MAE'] = iqr_based_mae(y_true, y_pred)
    metrics[f'{prefix}ME'] = calculate_max_error(y_true, y_pred)
    metrics[f'{prefix}R2'] = r2_score(y_true, y_pred)
    
    # Calculate CA (if R2 is valid)
    if not np.isnan(metrics[f'{prefix}R2']):
        metrics[f'{prefix}CA'] = calculate_ca(
            metrics[f'{prefix}RMSE'], 
            metrics[f'{prefix}MAE'], 
            metrics[f'{prefix}R2']
        )
    else:
        metrics[f'{prefix}CA'] = np.nan
    
    # Calculate MAPE
    metrics[f'{prefix}MAPE'] = safe_mape(y_true, y_pred)
    
    return metrics

# Horizontal and vertical feature indices
horizontal_indices = [0, 2, 4, 6, 8]
vertical_indices = [1, 3, 5, 7, 9]

# Helper function: Extract features by index and perform inverse normalization
def extract_and_inverse(preds_np, targets_np, indices, scaler, total_features=10):
    """Extract features by index and perform inverse normalization"""
    # Create full arrays
    full_preds = np.zeros((preds_np.shape[0], total_features))
    full_targets = np.zeros((targets_np.shape[0], total_features))
    
    # Place the extracted features back at the corresponding positions
    for i, idx in enumerate(indices):
        full_preds[:, idx] = preds_np[:, i]
        full_targets[:, idx] = targets_np[:, i]
    
    # Inverse normalization
    full_preds_inv = scaler.inverse_transform(full_preds)
    full_targets_inv = scaler.inverse_transform(full_targets)
    
    # Re-extract features
    preds_inv = np.zeros_like(preds_np)
    targets_inv = np.zeros_like(targets_np)
    
    for i, idx in enumerate(indices):
        preds_inv[:, i] = full_preds_inv[:, idx]
        targets_inv[:, i] = full_targets_inv[:, idx]
    
    return preds_inv, targets_inv


In [10]:
# ## 3. Data reading function

# %%
def ReadExcel(excelpath: str):
    DF = pd.read_excel(excelpath, header=None)    
    Values = DF.iloc[1:, 1:].values   
    Values = Values.astype('float32') 
    print(f"File {os.path.basename(excelpath)} shape: {Values.shape}")
    return Values

In [11]:
# ## 4. Load Scalers and Model
# Add joblib import at the beginning of the code
import joblib

# %%
# Load scalers
scaler_save_dir = './Scaler'
scaler_input = joblib.load(os.path.join(scaler_save_dir, 'scaler_input.pkl'))
scaler_output = joblib.load(os.path.join(scaler_save_dir, 'scaler_output.pkl'))
print("Scalers loaded successfully")

# Load model
model_path = './Model/Best_CNN_TSMixer_MA.pth'

# Need to import the model class first
from torchTSMixer.cnn_tsmixer_atten import TSMixerWithCNNAndAttention
# Assuming the model class is in a local file, adjust according to actual situation
# Here we use the method of loading the entire model directly
model = torch.load(model_path, map_location=device, weights_only=False)
model = model.to(device)
model.eval()
print("Model loaded successfully")
print(f"Model structure: {model}")

Scalers loaded successfully
Model loaded successfully
Model structure: TSMixerWithCNNAndAttention(
  (conv1): Conv1d(3, 3, kernel_size=(3,), stride=(1,), padding=same)
  (mixer_layers): Sequential(
    (0): MixerLayer(
      (time_mixing): TimeMixing(
        (norm): TimeBatchNorm2d(180, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (fc1): Linear(in_features=60, out_features=60, bias=True)
      )
      (feature_mixing): FeatureMixing(
        (norm_before): TimeBatchNorm2d(180, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (norm_after): Identity()
        (dropout): Dropout(p=0.2, inplace=False)
        (fc1): Linear(in_features=3, out_features=256, bias=True)
        (fc2): Linear(in_features=256, out_features=10, bias=True)
        (projection): Linear(in_features=3, out_features=10, bias=True)
      )
    )
  )
  (temporal_projection): Linear(in_features=60, out_features=1, bias=Tru

In [12]:
# ## 5. Load Test Data and Normalize

# %%
# List of test data files
test_data_files = [
    "Condition_1_32_sametime.xlsx",
    "Condition_9_23_600.xlsx", 
    "Condition_12_32_750.xlsx",
    "Condition_15_23_1050.xlsx",
    "Condition_22_32_1500.xlsx"
]

test_data_path = './TestData'

# Store data for each condition
test_data_list = []
test_condition_names = []

for file_name in test_data_files:
    condition_name = os.path.splitext(file_name)[0]
    print(f"\nProcessing condition: {condition_name}")
    
    # Read data
    file_path = os.path.join(test_data_path, file_name)
    data = ReadExcel(file_path)
    
    # Split input and output
    input_data = data[:, :input_data_num]  # First 180 columns
    output_data = data[:, input_data_num:]  # Last 10 columns
    
    # Reshape input data to (samples, 60, 3)
    input_data_reshaped = input_data.reshape(-1, hist_step, input_channel)
    
    # Normalize input data
    input_data_norm = scaler_input.transform(input_data_reshaped.reshape(-1, input_channel)).reshape(-1, hist_step, input_channel)
    
    # Normalize output data
    output_data_norm = scaler_output.transform(output_data)
    
    test_data_list.append((input_data_norm, output_data_norm))
    test_condition_names.append(condition_name)
    
    print(f"  Input data shape: {input_data_norm.shape}")
    print(f"  Output data shape: {output_data_norm.shape}")
    


Processing condition: Condition_1_32_sametime
File Condition_1_32_sametime.xlsx shape: (836, 190)
  Input data shape: (836, 60, 3)
  Output data shape: (836, 10)

Processing condition: Condition_9_23_600
File Condition_9_23_600.xlsx shape: (1056, 190)
  Input data shape: (1056, 60, 3)
  Output data shape: (1056, 10)

Processing condition: Condition_12_32_750
File Condition_12_32_750.xlsx shape: (1671, 190)
  Input data shape: (1671, 60, 3)
  Output data shape: (1671, 10)

Processing condition: Condition_15_23_1050
File Condition_15_23_1050.xlsx shape: (1508, 190)
  Input data shape: (1508, 60, 3)
  Output data shape: (1508, 10)

Processing condition: Condition_22_32_1500
File Condition_22_32_1500.xlsx shape: (2353, 190)
  Input data shape: (2353, 60, 3)
  Output data shape: (2353, 10)


In [13]:
# ## 6. Model Validation

# %%
# Store metrics for each condition
condition_horizontal_metrics = {}
condition_vertical_metrics = {}

# Create Excel writer object
excel_path = './CNN_TSMixer_MA_NewTest_Actual_Predicted.xlsx'

# Create Excel writer using openpyxl engine
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    
    for i, condition_name in enumerate(test_condition_names):
        print(f"\n{'='*60}")
        print(f"Validation results for Condition {condition_name}:")
        print(f"{'='*60}")
        
        # Get data for this condition
        x_test, y_test = test_data_list[i]
        x_tensor = torch.tensor(x_test, dtype=torch.float32).to(device)
        y_test_np = y_test
        
        # Make predictions using the model
        with torch.no_grad():
            pred_tensor = model(x_tensor)
            pred_tensor = pred_tensor.reshape(pred_tensor.shape[0], -1)
        
        pred_np = pred_tensor.cpu().numpy()
        
        # Inverse transform
        pred_inv = scaler_output.inverse_transform(pred_np)
        target_inv = scaler_output.inverse_transform(y_test_np)
        
        # Extract horizontal and vertical components
        horizontal_pred_inv = pred_inv[:, horizontal_indices]
        horizontal_target_inv = target_inv[:, horizontal_indices]
        vertical_pred_inv = pred_inv[:, vertical_indices]
        vertical_target_inv = target_inv[:, vertical_indices]
        
        # Calculate metrics
        condition_horizontal = calculate_all_metrics(horizontal_target_inv, horizontal_pred_inv, "Horizontal_")
        condition_vertical = calculate_all_metrics(vertical_target_inv, vertical_pred_inv, "Vertical_")
        
        # Store metrics
        condition_horizontal_metrics[condition_name] = condition_horizontal
        condition_vertical_metrics[condition_name] = condition_vertical
        
        # Print main metrics
        print(f"  Horizontal Velocity Metrics:")
        print(f"    MSE: {condition_horizontal['Horizontal_MSE']:.6f}, RMSE: {condition_horizontal['Horizontal_RMSE']:.6f}")
        print(f"    MAE: {condition_horizontal['Horizontal_MAE']:.6f}, MdAE: {condition_horizontal['Horizontal_MdAE']:.6f}")
        print(f"    IQR_MAE: {condition_horizontal['Horizontal_IQR_MAE']:.6f}, ME: {condition_horizontal['Horizontal_ME']:.6f}")
        print(f"    R²: {condition_horizontal['Horizontal_R2']:.6f}, CA: {condition_horizontal['Horizontal_CA']:.6f}")
        print(f"    MAPE: {condition_horizontal['Horizontal_MAPE']:.2f}%")
        
        print(f"  Vertical Velocity Metrics:")
        print(f"    MSE: {condition_vertical['Vertical_MSE']:.6f}, RMSE: {condition_vertical['Vertical_RMSE']:.6f}")
        print(f"    MAE: {condition_vertical['Vertical_MAE']:.6f}, MdAE: {condition_vertical['Vertical_MdAE']:.6f}")
        print(f"    IQR_MAE: {condition_vertical['Vertical_IQR_MAE']:.6f}, ME: {condition_vertical['Vertical_ME']:.6f}")
        print(f"    R²: {condition_vertical['Vertical_R2']:.6f}, CA: {condition_vertical['Vertical_CA']:.6f}")
        print(f"    MAPE: {condition_vertical['Vertical_MAPE']:.2f}%")
        
        # Save the simulated and predicted values for this condition to an Excel file
        data_list = []
        
        for sample_idx in range(len(target_inv)):
            sample_data = []
            # For the 5 measurement points
            for point_idx in range(5):
                # Get actual and predicted values for horizontal and vertical components
                u_actual = target_inv[sample_idx, horizontal_indices[point_idx]]
                v_actual = target_inv[sample_idx, vertical_indices[point_idx]]
                u_predicted = pred_inv[sample_idx, horizontal_indices[point_idx]]
                v_predicted = pred_inv[sample_idx, vertical_indices[point_idx]]
                
                sample_data.extend([u_actual, v_actual, u_predicted, v_predicted])
            
            data_list.append(sample_data)
        
        # Define column names
        columns = []
        for point_idx in range(1, 6):  # 1 to 5
            columns.extend([
                f'U_Actual_{point_idx}',
                f'V_Actual_{point_idx}',
                f'U_Predicted_{point_idx}',
                f'V_Predicted_{point_idx}'
            ])
        
        # Create DataFrame
        df = pd.DataFrame(data_list, columns=columns)
        
        # Save to the current sheet in Excel
        sheet_name = f'Condition_{condition_name}'
        if len(sheet_name) > 31:
            sheet_name = sheet_name[:31]
        
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"  Data saved to Excel sheet: {sheet_name}, total {len(df)} rows")

print(f"\nAll condition data saved to Excel file: {excel_path}")


Validation results for Condition Condition_1_32_sametime:
  Horizontal Velocity Metrics:
    MSE: 0.000781, RMSE: 0.027951
    MAE: 0.024302, MdAE: 0.023724
    IQR_MAE: 0.023729, ME: 0.060169
    R²: -1.151284, CA: 0.727167
    MAPE: 50.06%
  Vertical Velocity Metrics:
    MSE: 0.008115, RMSE: 0.090084
    MAE: 0.083486, MdAE: 0.087096
    IQR_MAE: 0.086311, ME: 0.196285
    R²: -1.631501, CA: 0.925673
    MAPE: 51.16%
  Data saved to Excel sheet: Condition_Condition_1_32_sameti, total 836 rows

Validation results for Condition Condition_9_23_600:
  Horizontal Velocity Metrics:
    MSE: 0.000092, RMSE: 0.009574
    MAE: 0.007900, MdAE: 0.007196
    IQR_MAE: 0.007193, ME: 0.025294
    R²: 0.192363, CA: 0.272287
    MAPE: 24.52%
  Vertical Velocity Metrics:
    MSE: 0.001037, RMSE: 0.032210
    MAE: 0.027639, MdAE: 0.026840
    IQR_MAE: 0.026563, ME: 0.080080
    R²: 0.064354, CA: 0.328513
    MAPE: 36.57%
  Data saved to Excel sheet: Condition_Condition_9_23_600, total 1056 rows

Vali

In [14]:
# ## 7. Overall assessment results

# %%
# Calculate overall test set metrics
print("\n" + "="*80)
print("Calculate overall test set metrics")
print("="*80)

# Merge data for all operating conditions
all_targets_inv = []
all_preds_inv = []
all_horizontal_targets_inv = []
all_horizontal_preds_inv = []
all_vertical_targets_inv = []
all_vertical_preds_inv = []

for i, condition_name in enumerate(test_condition_names):
    x_test, y_test = test_data_list[i]
    x_tensor = torch.tensor(x_test, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        pred_tensor = model(x_tensor)
        pred_tensor = pred_tensor.reshape(pred_tensor.shape[0], -1)
    
    pred_np = pred_tensor.cpu().numpy()
    y_test_np = y_test
    
    pred_inv = scaler_output.inverse_transform(pred_np)
    target_inv = scaler_output.inverse_transform(y_test_np)
    
    horizontal_pred_inv = pred_inv[:, horizontal_indices]
    horizontal_target_inv = target_inv[:, horizontal_indices]
    vertical_pred_inv = pred_inv[:, vertical_indices]
    vertical_target_inv = target_inv[:, vertical_indices]
    
    all_targets_inv.append(target_inv)
    all_preds_inv.append(pred_inv)
    all_horizontal_targets_inv.append(horizontal_target_inv)
    all_horizontal_preds_inv.append(horizontal_pred_inv)
    all_vertical_targets_inv.append(vertical_target_inv)
    all_vertical_preds_inv.append(vertical_pred_inv)

# Merge all data
all_targets_inv = np.vstack(all_targets_inv)
all_preds_inv = np.vstack(all_preds_inv)
all_horizontal_targets_inv = np.vstack(all_horizontal_targets_inv)
all_horizontal_preds_inv = np.vstack(all_horizontal_preds_inv)
all_vertical_targets_inv = np.vstack(all_vertical_targets_inv)
all_vertical_preds_inv = np.vstack(all_vertical_preds_inv)

# Calculation of the overall indicators
horizontal_metrics = calculate_all_metrics(all_horizontal_targets_inv, all_horizontal_preds_inv, "Tran_")
vertical_metrics = calculate_all_metrics(all_vertical_targets_inv, all_vertical_preds_inv, "Long_")

print(f"\n Overall test set performance:")
print(f"  Samples in test set: {len(all_targets_inv)}")

print(f"\n  Performance of transverse flow velocity:")
print(f"    MSE: {horizontal_metrics['Tran_MSE']:.6f}")
print(f"    RMSE: {horizontal_metrics['Tran_RMSE']:.6f}")
print(f"    MAE: {horizontal_metrics['Tran_MAE']:.6f}")
print(f"    MdAE: {horizontal_metrics['Tran_MdAE']:.6f}")
print(f"    IQR_MAE: {horizontal_metrics['Tran_IQR_MAE']:.6f}")
print(f"    ME: {horizontal_metrics['Tran_ME']:.6f}")
print(f"    R²: {horizontal_metrics['Tran_R2']:.6f}")
print(f"    CA: {horizontal_metrics['Tran_CA']:.6f}")
print(f"    MAPE: {horizontal_metrics['Tran_MAPE']:.2f}%")

print(f"\n  Performance of longitudinal flow velocity:")
print(f"    MSE: {vertical_metrics['Long_MSE']:.6f}")
print(f"    RMSE: {vertical_metrics['Long_RMSE']:.6f}")
print(f"    MAE: {vertical_metrics['Long_MAE']:.6f}")
print(f"    MdAE: {vertical_metrics['Long_MdAE']:.6f}")
print(f"    IQR_MAE: {vertical_metrics['Long_IQR_MAE']:.6f}")
print(f"    ME: {vertical_metrics['Long_ME']:.6f}")
print(f"    R²: {vertical_metrics['Long_R2']:.6f}")
print(f"    CA: {vertical_metrics['Long_CA']:.6f}")
print(f"    MAPE: {vertical_metrics['Long_MAPE']:.2f}%")

# Save the overall assessment results
overall_df = pd.DataFrame({
    'Types of indicators': ['Tran'] * len(horizontal_metrics) + ['Long'] * len(vertical_metrics),
    'Name of indicators': list(horizontal_metrics.keys()) + list(vertical_metrics.keys()),
    'Indicator value': list(horizontal_metrics.values()) + list(vertical_metrics.values())
})

overall_results_path = './Overall assessment results.csv' 
overall_df.to_csv(overall_results_path, index=False, encoding='utf-8-sig')
print(f"\n The overall assessment results have been saved to: {overall_results_path}")

# Save the result of sub-condition evaluation
condition_dfs = []
for condition_name in test_condition_names:
    # Tran_indicators
    for key, value in condition_horizontal_metrics[condition_name].items():
        condition_dfs.append({
            'Condition': condition_name,
            'Types of indicators': 'Tran',
            'Name of indicators': key,
            'Indicator value': value
        })
    
    # Long_indicators
    for key, value in condition_vertical_metrics[condition_name].items():
        condition_dfs.append({
            'Condition': condition_name,
            'Types of indicators': 'Long',
            'Name of indicators': key,
            'Indicator value': value
        })

condition_df = pd.DataFrame(condition_dfs)
condition_results_path = './Evaluation results of different conditions.csv' 
condition_df.to_csv(condition_results_path, index=False, encoding='utf-8-sig')
print(f"The sub-condition evaluation results have been saved to: {condition_results_path}")

print("\n" + "="*80)
print("Model validation complete!")
print("="*80)


Calculate overall test set metrics

 Overall test set performance:
  Samples in test set: 7424

  Performance of transverse flow velocity:
    MSE: 0.000276
    RMSE: 0.016608
    MAE: 0.013298
    MdAE: 0.011368
    IQR_MAE: 0.011264
    ME: 0.060169
    R²: 0.087779
    CA: 0.310902
    MAPE: 40.55%

  Performance of longitudinal flow velocity:
    MSE: 0.004157
    RMSE: 0.064477
    MAE: 0.054495
    MdAE: 0.050676
    IQR_MAE: 0.050928
    ME: 0.196285
    R²: -0.040829
    CA: 0.382734
    MAPE: 136.18%

 The overall assessment results have been saved to: ./Overall assessment results.csv
The sub-condition evaluation results have been saved to: ./Evaluation results of different conditions.csv

Model validation complete!
